# Create grid where each AIS point as centroid of a grid cell. Have it match with GFW grid schema. Convert to UTM Zone 19N. 

## 1. Set Environment

In [14]:
import arcpy
from arcpy import env

env.workspace = r"C:\\Users\\mmccaffrey17\\ArcGIS\\Projects\\VesselMapping_SNE\\VesselMapping_SNE.gdb"
env.overwriteOutput = True

## 2. Input Point Feature Class

In [15]:
gfw_vp_pts = r"C:\\Users\\mmccaffrey17\\ArcGIS\\Projects\\VesselMapping_SNE\\gfw_vp.shp"

# confirm spatial ref

desc = arcpy.Describe(gfw_vp_pts)
print(desc.spatialReference.name)

GCS_WGS_1984


## 3. Create a Unique Point Feature Class

In [16]:
unique_pts = "gfw_vp_unique_pts"

arcpy.management.DeleteIdentical(
    in_dataset=gfw_vp_pts,
    fields=["Lon", "Lat"]
)

arcpy.management.CopyFeatures(
    in_features=gfw_vp_pts,
    out_feature_class=unique_pts
)

<Result 'C:\\\\Users\\\\mmccaffrey17\\\\ArcGIS\\\\Projects\\\\VesselMapping_SNE\\\\VesselMapping_SNE.gdb\\gfw_vp_unique_pts'>

## 4. Create Empty Polygon Feature Class

In [17]:
AOI_sqGrid = "AOI_sqGridWGS"

arcpy.management.CreateFeatureclass(
    out_path=env.workspace,
    out_name=AOI_sqGrid,
    geometry_type="POLYGON",
    spatial_reference=gfw_vp_pts
)

# carry cell centroids over
arcpy.management.AddField(AOI_sqGrid, "Lon_C", "DOUBLE")
arcpy.management.AddField(AOI_sqGrid, "Lat_C", "DOUBLE")

<Result 'C:\\Users\\mmccaffrey17\\ArcGIS\\Projects\\VesselMapping_SNE\\VesselMapping_SNE.gdb\\AOI_sqGridWGS'>

## 5. Construct 0.01 Degree Square Around Each Point

In [18]:
half = 0.005  # half of 0.01 degrees

with arcpy.da.SearchCursor(unique_pts, ["SHAPE@XY", "Lon", "Lat"]) as scur, \
     arcpy.da.InsertCursor(AOI_sqGrid, ["SHAPE@", "Lon_C", "Lat_C"]) as icur:

    for (x, y), lon, lat in scur:

        array = arcpy.Array([
            arcpy.Point(x - half, y - half),  # lower left
            arcpy.Point(x - half, y + half),  # upper left
            arcpy.Point(x + half, y + half),  # upper right
            arcpy.Point(x + half, y - half),  # lower right
            arcpy.Point(x - half, y - half)   # close polygon
        ])

        polygon = arcpy.Polygon(array, arcpy.Describe(unique_pts).spatialReference)

        icur.insertRow([polygon, lon, lat])


## 6. Validate Geometry

In [19]:
# calculate polygon centroids

arcpy.management.FeatureToPoint(
    in_features=AOI_sqGrid,
    out_feature_class="grid_centroids",
    point_location="CENTROID"
)

<Result 'C:\\\\Users\\\\mmccaffrey17\\\\ArcGIS\\\\Projects\\\\VesselMapping_SNE\\\\VesselMapping_SNE.gdb\\grid_centroids'>

In [20]:
# use near analysis to measure offset

arcpy.analysis.Near(
    in_features="grid_centroids",
    near_features=unique_pts
) # adds a NEAR_DIST field

<Result 'C:\\Users\\mmccaffrey17\\ArcGIS\\Projects\\VesselMapping_SNE\\VesselMapping_SNE.gdb\\grid_centroids'>

In [21]:
# check for errors

with arcpy.da.SearchCursor("grid_centroids", ["NEAR_DIST"]) as cursor:
    max_dist = max(row[0] for row in cursor)

print(f"Maximum centroid-point distance: {max_dist}") # e-15 value is a safe floating-point artifact

Maximum centroid-point distance: 1.6077746776921858e-13


Good to move on to UTM projection!

## 7. Define inputs and outputs

In [22]:
# AOI_sqGridWGS already defined
# gfw_vp_pts already defined

AOI_sqGrid_UTM = "AOI_sqGrid_UTM" # output polygon
gfw_vp_UTM = "gfw_vp_UTM" # output point

## 8. Verify Source Spatial Reference

In [23]:
for fc in [AOI_sqGrid, gfw_vp_pts]:
    sr = arcpy.Describe(fc).spatialReference

    print(fc, sr.name, sr.factoryCode)

AOI_sqGridWGS GCS_WGS_1984 4326
C:\\Users\\mmccaffrey17\\ArcGIS\\Projects\\VesselMapping_SNE\\gfw_vp.shp GCS_WGS_1984 4326


## 9. Create Target Spatial Reference (UTM 19N)

In [24]:
utm19n = arcpy.SpatialReference(32619)

## 10. Project the Polygon Grid

In [25]:
arcpy.management.Project(
    in_dataset=AOI_sqGrid,
    out_dataset=AOI_sqGrid_UTM,
    out_coor_system=utm19n,
    transform_method="", # not required for WGS84 to WGS 84
    preserve_shape="PRESERVE_SHAPE"
)

<Result 'C:\\\\Users\\\\mmccaffrey17\\\\ArcGIS\\\\Projects\\\\VesselMapping_SNE\\\\VesselMapping_SNE.gdb\\AOI_sqGrid_UTM'>

## 11. Project the Point Feature Class

In [26]:

arcpy.management.Project(
    in_dataset=gfw_vp_pts,
    out_dataset=gfw_vp_UTM,
    out_coor_system=utm19n,
    transform_method=""
)

<Result 'C:\\\\Users\\\\mmccaffrey17\\\\ArcGIS\\\\Projects\\\\VesselMapping_SNE\\\\VesselMapping_SNE.gdb\\gfw_vp_UTM'>

## 12. Validate Geometry

In [27]:
# double check projections

for fc in [AOI_sqGrid_UTM, gfw_vp_UTM]:
    sr = arcpy.Describe(fc).spatialReference

    print(fc, sr.name, sr.linearUnitName)

AOI_sqGrid_UTM WGS_1984_UTM_Zone_19N Meter
gfw_vp_UTM WGS_1984_UTM_Zone_19N Meter


In [28]:
##  confirm centroid-to-centroid alignment

# create polygon centroids

arcpy.management.FeatureToPoint(
    in_features=AOI_sqGrid_UTM,
    out_feature_class="grid_utm_centroids",
    point_location="CENTROID"
)

# measure distance to AIS points

arcpy.analysis.Near(
    in_features="grid_utm_centroids",
    near_features=gfw_vp_UTM
)

# inspect max offset

with arcpy.da.SearchCursor(
    "grid_utm_centroids", ["NEAR_DIST"]
) as cursor:
    max_dist = max(row[0] for row in cursor)

print(f"Maximum centroid‑to‑point offset (meters): {max_dist}")

Maximum centroid‑to‑point offset (meters): 0.0020024988048917133


## 13. Calculate Cell Area of Projected Grid Polygons

In [29]:
# add area field

Area = "Area_m2"

if Area not in [f.name for f in arcpy.ListFields(AOI_sqGrid_UTM)]:
    arcpy.management.AddField(
        AOI_sqGrid_UTM,
        Area,
        "DOUBLE"
    )

In [30]:
# calculate area using shape geometry

arcpy.management.CalculateGeometryAttributes(
    in_features=AOI_sqGrid_UTM,
    geometry_property=[[Area, "AREA"]],
    area_unit="SQUARE_METERS"
)


<Result 'C:\\Users\\mmccaffrey17\\ArcGIS\\Projects\\VesselMapping_SNE\\VesselMapping_SNE.gdb\\AOI_sqGrid_UTM'>

In [ ]:
# validate calculations

areas = [row[0] for row in arcpy.da.SearchCursor(AOI_sqGrid_UTM, [Area])]

print(f"Min area (m²): {min(areas)}")
print(f"Max area (m²): {max(areas)}")

# add area in sq km

arcpy.management.AddField(AOI_sqGrid_UTM, "Area_km2", "DOUBLE")

arcpy.management.CalculateField(
    AOI_sqGrid_UTM,
    "Area_km2",
    f"!{Area}! / 1e6",
    "PYTHON3"
)

Min area (m²): 928939.7357001425
Max area (m²): 934806.316239057


<Result 'C:\\Users\\mmccaffrey17\\ArcGIS\\Projects\\VesselMapping_SNE\\VesselMapping_SNE.gdb\\AOI_sqGrid_UTM'>

In [33]:
# confirm grid cell consistency

import statistics

mean_area = statistics.mean(areas)
stdev_area = statistics.stdev(areas)

print(f"Mean area: {mean_area:.2f} m²")
print(f"Std deviation: {stdev_area:.2f} m²")

Mean area: 931984.65 m²
Std deviation: 1485.12 m²


good to spatial join points and grid cells then move onto spatial analyses!